# 05-07 RunnableWithMessageHistory: 대화 기록 관리하기

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있어요:

- `RunnableWithMessageHistory`로 체인에 세션별 대화 기록을 추가하는 방법을 구현할 수 있어요
- `MessagesPlaceholder`의 역할을 이해하고, `history_messages_key`와 변수명을 올바르게 연결할 수 있어요
- 같은 세션 내에서 이전 대화 맥락이 유지되고, 다른 세션과 대화가 섞이지 않는 것을 확인할 수 있어요
- `ConfigurableFieldSpec`으로 사용자 ID와 대화 ID를 조합한 세션 키를 설계할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 다음 개념을 알고 있으면 좋아요:

- `ChatPromptTemplate`과 메시지 역할(system/human/ai) 이해
- LCEL 파이프 연산자(`|`) 사용법
- 파이썬 딕셔너리와 클로저(Closure) 기초

---

`RunnableWithMessageHistory`를 사용하면 체인에 메시지 기록(대화 이력)을 추가할 수 있어요.

**주요 특징:**
- 세션(Session)별 대화 기록 관리
- 이전 대화 맥락을 유지하여 연속적인 대화 가능
- 인메모리(In-Memory) 또는 영구 저장소(Redis 등) 지원

**활용 사례:**
- 대화형 챗봇(Chatbot) 개발
- 복잡한 데이터 처리에서 이전 단계 결과 참조
- 상태 관리가 필요한 애플리케이션

In [ ]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


In [ ]:
# ============================================================
# TODO: MessagesPlaceholder를 포함한 프롬프트와 기본 체인을 완성하세요
# 힌트: MessagesPlaceholder(variable_name="history") — 대화 기록이 삽입될 위치
# 예상 결과: "✅ 기본 체인 생성 완료" 출력
# ============================================================

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_openai import ChatOpenAI

# ---------------------------------------------------
# MessagesPlaceholder: 대화 기록을 프롬프트에 삽입하는 자리표시자
# ---------------------------------------------------

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")

# 대화 기록을 포함한 프롬프트 템플릿
# MessagesPlaceholder: 이 위치에 세션별 대화 기록이 순서대로 삽입됨
# - variable_name 값은 RunnableWithMessageHistory의 history_messages_key와 반드시 일치해야 함


# 기본 체인 생성 (히스토리 없이 — 다음 단계에서 RunnableWithMessageHistory로 감쌈)


## MessagePlaceholder 이해하기

`MessagesPlaceholder`는 대화 기록처럼 **여러 메시지를 한 번에 삽입**할 때 사용하는 LangChain 구성 요소입니다. `RunnableWithMessageHistory`와 함께 쓰면, 세션별로 누적된 메시지를 프롬프트 안에 자연스럽게 끼워넣을 수 있습니다.

### 왜 필요할까요?
- 체인이 실행될 때마다 직전에 오간 대화를 **전달**해야 LLM이 맥락을 기억합니다.
- `MessagesPlaceholder(variable_name="history")`는 해당 변수 이름으로 전달된 메시지 목록을 그대로 추가해 줍니다.
- `RunnableWithMessageHistory`에서 `history_messages_key="history"`로 지정하면, 프롬프트에 같은 이름의 `MessagesPlaceholder`가 있어야 합니다.

### 사용 패턴
```python
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {ability}에 능숙한 어시스턴트입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])
```
- `system` 메시지 → 기본 지침
- `MessagesPlaceholder` → 세션별 대화 기록이 순서대로 삽입
- `human` 메시지 → 이번 호출의 최신 사용자 입력

💡 **TIP:** `variable_name` 값과 `RunnableWithMessageHistory`의 `history_messages_key` 값이 다르면 기록이 프롬프트로 전달되지 않습니다. 두 값을 항상 일치시키세요.


## 1. 인메모리 대화 기록 (In-Memory)

가장 간단한 방법으로, 대화 기록을 메모리에 저장해요.

**특징:**
- 빠른 접근 속도
- 애플리케이션 재시작 시 기록 소실
- 개발/테스트에 적합

> **주의:** `MessagesPlaceholder(variable_name="history")`의 `variable_name` 값과 `RunnableWithMessageHistory`의 `history_messages_key` 값이 반드시 일치해야 해요. 두 값이 다르면 대화 기록이 프롬프트에 전달되지 않아요.

In [ ]:
# ============================================================
# TODO: get_session_history 함수를 작성하고 RunnableWithMessageHistory를 생성하세요
# 힌트: session_id를 키로 store 딕셔너리에서 ChatMessageHistory 반환
#       RunnableWithMessageHistory(runnable, get_session_history, input_messages_key="input", history_messages_key="history")
# 예상 결과: 세션별 독립 대화 기록 관리
# ============================================================

# ---------------------------------------------------
# RunnableWithMessageHistory: 체인에 세션별 대화 기록 추가
# ---------------------------------------------------


In [ ]:
# 첫 번째 대화
# 
# 중요: config에 session_id를 반드시 지정해야 합니다


In [ ]:
# 두 번째 대화 (같은 세션)
# 
# 이전 대화 내용을 기억하고 연속적으로 대화 가능


In [ ]:
# 다른 세션으로 대화
# 
# 다른 session_id를 사용하면 새로운 대화가 시작됨


## 2. 사용자 정의 세션 키 사용

기본 `session_id` 대신 사용자 정의 키를 사용할 수 있어요.

예를 들어, `user_id`와 `conversation_id`를 함께 사용하여 더 세밀한 세션 관리가 가능해요. 이 방식은 멀티 유저(Multi-User) 서비스에서 유용해요.

> **실무 팁:** 프로덕션(Production) 환경에서는 세션 스토어로 `RedisChatMessageHistory`를 사용하는 것을 권장해요. 서버가 재시작되어도 대화 기록이 유지되고, 여러 서버 인스턴스 간에 세션을 공유할 수 있어요.

In [ ]:
from langchain_core.runnables import ConfigurableFieldSpec

# ============================================================
# TODO: user_id와 conversation_id를 조합하는 사용자 정의 세션 키를 설정하세요
# 힌트: ConfigurableFieldSpec(id="user_id", ...) + ConfigurableFieldSpec(id="conversation_id", ...)
# 예상 결과: config={"configurable": {"user_id": "...", "conversation_id": "..."}}으로 세션 구분
# ============================================================

# ---------------------------------------------------
# ConfigurableFieldSpec: 사용자 정의 세션 키 설계
# ---------------------------------------------------


In [ ]:
# 사용자 정의 세션 키로 대화


In [ ]:
# Redis를 사용한 대화 기록 (선택사항)
# 
# 주의: Redis 서버가 실행 중이어야 합니다
# 
# Redis 사용 예제:
# from langchain_community.chat_message_histories import RedisChatMessageHistory
# 
# REDIS_URL = "redis://localhost:6379/0"
# 
# def get_redis_history(session_id: str) -> RedisChatMessageHistory:
#     return RedisChatMessageHistory(session_id, url=REDIS_URL)
# 
# with_redis_history = RunnableWithMessageHistory(
#     runnable,
#     get_redis_history,
#     input_messages_key="input",
#     history_messages_key="history",
# )

print("=" * 60)
print("📝 Redis 사용 예제 (주석 처리됨)")
print("=" * 60)
print("Redis를 사용하려면:")
print("1. pip install redis")
print("2. Redis 서버 실행")
print("3. 위 주석을 해제하고 사용")


## 마무리 요약

### RunnableWithMessageHistory 주요 파라미터

| 파라미터 | 역할 | 예시 |
|----------|------|------|
| `runnable` | 실행할 기본 체인 | `prompt \| model` |
| `get_session_history` | 세션 ID로 기록 반환하는 함수 | `lambda sid: store[sid]` |
| `input_messages_key` | 입력 메시지의 딕셔너리 키 | `"input"` |
| `history_messages_key` | 대화 기록의 딕셔너리 키 | `"history"` |
| `history_factory_config` | 사용자 정의 세션 키 설정 | `[ConfigurableFieldSpec(...)]` |

### 핵심 요점

- `MessagesPlaceholder`의 `variable_name`과 `history_messages_key`를 반드시 일치시켜야 해요
- `session_id`가 다르면 완전히 별도의 대화 기록이 유지돼요
- 인메모리 방식은 개발 환경에, Redis 방식은 프로덕션 환경에 적합해요
- `ConfigurableFieldSpec`으로 user_id + conversation_id 조합 등 세밀한 세션 설계가 가능해요

### 다음 노트북 예고

다음 노트북(`08-Custom-Generator.ipynb`)에서는 파이썬 제너레이터(Generator) 함수를 LCEL 파이프라인에 연결하여 스트리밍(Streaming) 방식으로 데이터를 실시간으로 처리하는 방법을 배워요.